In [ ]:
import ast
from dataclasses import dataclass, field
import itertools
import heapq
from typing import NamedTuple

import pandas as pd
import numpy as np

In [ ]:
import jax
from jax import jit
import jax.numpy as jnp
from jax.scipy.spatial.transform import Rotation

In [ ]:
import matplotlib.pyplot as plt

def plot_trajectories(measured_traj: np.ndarray, simulated_traj: np.ndarray = None):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    ax.plot(measured_traj[:, 0], measured_traj[:, 1], measured_traj[:, 2], color='blue', label='Measured Trajectory')
    if simulated_traj is not None:
        ax.plot(simulated_traj[:, 0], simulated_traj[:, 1], simulated_traj[:, 2], color='red', label='Simulated Trajectory')

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.legend()

    plt.show()

In [ ]:
# IDX_Q = slice(0, 4)
# IDX_P = slice(4, 7)
# IDX_O = slice(7, 10)
# IDX_V = slice(10, 13)
# IDX_BATTERY = slice(13, 14)
# IDX_TRIM = slice(14, 15)
# IDX_ACTUAL_THRUST = slice(15, 16)
#
# class SystemParams(NamedTuple):
#     MASS: float
#     G_WORLD: jnp.ndarray
#     I_TENSOR_DIAGONAL: jnp.ndarray
#     DRAG: jnp.ndarray
#     THRUST_CONSTANT: jnp.ndarray
#     MAX_THRUST: jnp.ndarray
#     GROUND_EFFECT_MAX: float
#     ROTOR_RADIUS: float
#     MAX_TAIL_THRUST: jnp.ndarray
#     TAIL_MOMENT_ARM: float
#     YAW_TIME_CONSTANT: float
#     GYRO_SPRING_CONSTANT: jnp.ndarray
#     ANGULAR_DRAG: jnp.ndarray
#     CORIOLIS_CONSTANT: float
#     ROTOR_TIME_CONSTANT: float
#
#     def __repr__(self):
#             args = []
#             for key, val in self._asdict().items():
#                 if hasattr(val, "tolist"):
#                     args.append(f"{key}=jnp.array({val.tolist()})")
#                 else:
#                     args.append(f"{key}={repr(val)}")
#
#             formatted_args = ",\n    ".join(args)
#             return f"{self.__class__.__name__}(\n    {formatted_args}\n)"
#
# @jit
# def propagate(s, dt, params: SystemParams, commands, ground=False):
#     pos_old = s[IDX_P]
#     vel_old = s[IDX_V]
#     omega_old = s[IDX_O]
#     quat_old = Rotation.from_quat(s[IDX_Q])
#     battery = s[IDX_BATTERY]
#     trim_old = s[IDX_TRIM]
#     actual_thrust_old = s[IDX_ACTUAL_THRUST]
#
#     I_TENSOR = jnp.diag(params.I_TENSOR_DIAGONAL)
#     I_INVERSE = jnp.diag(1 / params.I_TENSOR_DIAGONAL)
#
#
#     thrust, pitch, yaw = commands
#     thrust = jnp.sqrt(thrust)
#
#     actual_thrust_new = (actual_thrust_old +
#     ((thrust - actual_thrust_old) / params.ROTOR_TIME_CONSTANT) * dt)
#
#     # Orientation
#     dq = Rotation.from_rotvec(omega_old * dt)
#     quat_new = quat_old * dq
#
#     # Velocity
#     gravity = quat_new.inv().apply(params.G_WORLD * params.MASS)
#     alt = jnp.maximum(pos_old[2], 0.001)
#     ge_multiplier = 1.0 + params.GROUND_EFFECT_MAX * jnp.exp(-3.0 * (alt / params.ROTOR_RADIUS))
#
#     main_rotor_thrust = (params.THRUST_CONSTANT + actual_thrust_new * params.MAX_THRUST * battery) * ge_multiplier
#     tail_rotor_thrust = pitch * params.MAX_TAIL_THRUST * battery
#     thrust_vector = main_rotor_thrust + tail_rotor_thrust
#
#     drag = params.DRAG * vel_old
#     F_net_body = gravity + thrust_vector - drag
#     F_net_world_frame = quat_new.apply(F_net_body)
#
#     normal_z_mag = jnp.where(F_net_world_frame[2] < 0, -F_net_world_frame[2], 0.0)
#     normal = jnp.where(ground,
#     quat_new.inv().apply(jnp.array([0.0, 0.0, normal_z_mag])),
#     jnp.array([0.0, 0.0, 0.0]))
#
#     F_net = F_net_body + normal
#     acc_coriolis = jnp.cross(omega_old, vel_old)
#
#     vel_new = vel_old + ((F_net / params.MASS) - acc_coriolis) * dt
#
#     vel_new_world = quat_new.apply(vel_new)
#
#     is_sinking = ground & (vel_new_world[2] < 0.0)
#     vel_new_world = jnp.where(is_sinking, vel_new_world.at[2].set(0.0), vel_new_world)
#     vel_new = jnp.where(is_sinking, quat_new.inv().apply(vel_new_world), vel_new)
#
#     # Position
#     pos_new = pos_old + vel_new_world * dt
#     pos_new = jnp.where(ground & (pos_new[2] <= 0.0), pos_new.at[2].set(0.0), pos_new)
#
#     # Angular Velocity
#     tau_roll = acc_coriolis[1] * params.MASS * params.CORIOLIS_CONSTANT
#     tau_actuator_pitch = (params.MAX_TAIL_THRUST[2] * pitch * battery * params.TAIL_MOMENT_ARM).squeeze()
#     tau_actuator_yaw = ((I_TENSOR[2, 2] / params.YAW_TIME_CONSTANT) * jnp.clip(yaw + trim_old, -1.0, 1.0) * battery).squeeze()
#     tau_actuator = jnp.array([tau_roll, tau_actuator_pitch, tau_actuator_yaw])
#
#     tau_gyro = params.GYRO_SPRING_CONSTANT * quat_old.as_euler(seq='XYZ', degrees=False)
#     tau_damping = params.ANGULAR_DRAG * omega_old
#
#     tau_net = tau_actuator - tau_gyro - tau_damping
#
#     omega_cross_I_omega = jnp.cross(omega_old, I_TENSOR @ omega_old)
#     d_omega = I_INVERSE @ (tau_net - omega_cross_I_omega)
#
#     omega_new = omega_old + d_omega * dt
#
#     # Battery
#     battery_new = battery - (thrust * (dt / 360.))
#
#     s_new = jnp.concatenate([
#     quat_new.as_quat(canonical=True),
#     pos_new,
#     omega_new,
#     vel_new,
#     battery_new,
#     trim_old,
#     actual_thrust_new
#     ])
#
#     return s_new

In [ ]:
IDX_Q = slice(0, 4)
IDX_P = slice(4, 7)
IDX_O = slice(7, 10)
IDX_V = slice(10, 13)
IDX_BATTERY = slice(13, 14)
IDX_TRIM = slice(14, 15)
IDX_ACTUAL_COMMANDS = slice(15, 18)

class SystemParams(NamedTuple):
    MASS: float
    G_WORLD: jnp.ndarray
    I_TENSOR_DIAGONAL: jnp.ndarray
    DRAG: jnp.ndarray
    THRUST_CONSTANT: jnp.ndarray
    MAX_THRUST: jnp.ndarray
    GROUND_EFFECT_MAX: float
    ROTOR_RADIUS: float
    MAX_TAIL_THRUST: jnp.ndarray
    TAIL_MOMENT_ARM: float
    GYRO_SPRING_CONSTANT: jnp.ndarray
    ANGULAR_DRAG: jnp.ndarray
    CORIOLIS_CONSTANT: float
    ROTOR_TIME_CONSTANT: float
    PITCH_TIME_CONSTANT: float
    YAW_TIME_CONSTANT: float

    def __repr__(self):
            args = []
            for key, val in self._asdict().items():
                if hasattr(val, "tolist"):
                    args.append(f"{key}=jnp.array({val.tolist()})")
                else:
                    args.append(f"{key}={repr(val)}")

            formatted_args = ",\n    ".join(args)
            return f"{self.__class__.__name__}(\n    {formatted_args}\n)"

@jit
def propagate(s, dt, params: SystemParams, commands, ground=False):
    pos_old = s[IDX_P]
    vel_old = s[IDX_V]
    omega_old = s[IDX_O]
    quat_old = Rotation.from_quat(s[IDX_Q])
    battery = s[IDX_BATTERY]
    trim_old = s[IDX_TRIM]
    actual_commands_old = s[IDX_ACTUAL_COMMANDS]

    I_TENSOR = jnp.diag(params.I_TENSOR_DIAGONAL)
    I_INVERSE = jnp.diag(1 / params.I_TENSOR_DIAGONAL)


    # First order model for commands
    actual_thrust_old, actual_pitch_old, actual_yaw_old = actual_commands_old
    thrust, pitch, yaw = commands
    thrust = jnp.sqrt(thrust)

    actual_thrust_new = (actual_thrust_old +
                         ((thrust - actual_thrust_old) / params.ROTOR_TIME_CONSTANT) * dt)

    actual_pitch_new = (actual_pitch_old +
                         ((pitch - actual_pitch_old) / params.PITCH_TIME_CONSTANT) * dt)

    # Mirrors remote control logic for adding yaw + trim
    yaw_cmd = jnp.clip(yaw + ((trim_old - 0.5) / 3), -1.0, 1.0)
    actual_yaw_new = (actual_yaw_old +
                      ((yaw_cmd - actual_yaw_old) / params.YAW_TIME_CONSTANT) * dt).squeeze()


    # Orientation
    dq = Rotation.from_rotvec(omega_old * dt)
    quat_new = quat_old * dq

    # Velocity
    gravity = quat_new.inv().apply(params.G_WORLD * params.MASS)
    alt = jnp.maximum(pos_old[2], 0.001)
    ge_multiplier = 1.0 + params.GROUND_EFFECT_MAX * jnp.exp(-3.0 * (alt / params.ROTOR_RADIUS))

    main_rotor_thrust = (params.THRUST_CONSTANT + actual_thrust_new * params.MAX_THRUST * battery) * ge_multiplier
    tail_rotor_thrust = actual_pitch_new * params.MAX_TAIL_THRUST * battery
    thrust_vector = main_rotor_thrust + tail_rotor_thrust

    drag = params.DRAG * vel_old
    F_net_body = gravity + thrust_vector - drag
    F_net_world_frame = quat_new.apply(F_net_body)

    normal_z_mag = jnp.where(F_net_world_frame[2] < 0, -F_net_world_frame[2], 0.0)
    normal = jnp.where(ground,
                       quat_new.inv().apply(jnp.array([0.0, 0.0, normal_z_mag])),
                       jnp.array([0.0, 0.0, 0.0]))

    F_net = F_net_body + normal
    acc_coriolis = jnp.cross(omega_old, vel_old)

    vel_new = vel_old + ((F_net / params.MASS) - acc_coriolis) * dt

    vel_new_world = quat_new.apply(vel_new)

    is_sinking = ground & (vel_new_world[2] < 0.0)
    vel_new_world = jnp.where(is_sinking, vel_new_world.at[2].set(0.0), vel_new_world)
    vel_new = jnp.where(is_sinking, quat_new.inv().apply(vel_new_world), vel_new)

    # Position
    pos_new = pos_old + vel_new_world * dt
    pos_new = jnp.where(ground & (pos_new[2] <= 0.0), pos_new.at[2].set(0.0), pos_new)

    # Angular Velocity
    tau_roll = acc_coriolis[1] * params.MASS * params.CORIOLIS_CONSTANT
    tau_actuator_pitch = (params.MAX_TAIL_THRUST[2] * pitch * battery * params.TAIL_MOMENT_ARM).squeeze()
    tau_actuator_yaw = ((I_TENSOR[2, 2] / params.YAW_TIME_CONSTANT) * jnp.clip(yaw + trim_old, -1.0, 1.0) * battery).squeeze()
    tau_actuator = jnp.array([tau_roll, tau_actuator_pitch, tau_actuator_yaw])

    tau_gyro = params.GYRO_SPRING_CONSTANT * quat_old.as_euler(seq='XYZ', degrees=False)
    tau_damping = params.ANGULAR_DRAG * omega_old

    tau_net = tau_actuator - tau_gyro - tau_damping

    omega_cross_I_omega = jnp.cross(omega_old, I_TENSOR @ omega_old)
    d_omega = I_INVERSE @ (tau_net - omega_cross_I_omega)

    omega_new = omega_old + d_omega * dt

    # Battery
    battery_new = battery - (thrust * (dt / 300.))

    s_new = jnp.concatenate([
        quat_new.as_quat(canonical=True),
        pos_new,
        omega_new,
        vel_new,
        battery_new,
        trim_old,
        jnp.array([actual_thrust_new, actual_pitch_new, actual_yaw_new]),
    ])

    return s_new

In [ ]:
df = pd.read_csv('./flight_recordings/throttle_tuning.csv')
df['command'] = df['command'].apply(lambda x: np.array(ast.literal_eval(x)))
df['position'] = df['position'].apply(lambda x: np.array(ast.literal_eval(x)))

In [ ]:
initial_state = jnp.array([          0,           0,     0.71842,     0.69561,  -0.0080318,    -0.10679,           0,           0,           0,           0,           0,           0,           0,           1.0,           0,           0, 0, 0])

# pitch tuning:
# initial_state = jnp.array([          0,           0,     0.74054,     0.67201,   -0.013604,    -0.49292,           0,           0,           0,           0,           0,           0,           0,           1.0,           0,           0, 0, 0])

In [ ]:
len(initial_state)

In [ ]:
def run_simulation(_initial_state, _timestamps, params, commands_seq, sim_fps=500):
    dts = jnp.diff(_timestamps)
    commands_to_apply = commands_seq[:-1]
    sim_dt = 1.0 / sim_fps

    def outer_step_fn(state, inputs):
        interval_dt, _commands = inputs
        inner_init_val = (state, 0.0)

        def cond_fn(inner_val):
            _, t = inner_val
            return (t + sim_dt) < interval_dt

        def body_fn(inner_val):
            current_state, t = inner_val
            _ground = current_state[IDX_P][2] <= 0.0
            next_state = propagate(current_state, sim_dt, params, _commands, ground=_ground)
            return next_state, t + sim_dt

        final_inner_state, time_simulated = jax.lax.while_loop(cond_fn, body_fn, inner_init_val)
        remainder_dt = interval_dt - time_simulated
        ground = final_inner_state[IDX_P][2] <= 0.0
        exact_state = propagate(final_inner_state, remainder_dt, params, _commands, ground=ground)

        return exact_state, exact_state

    _, states_history = jax.lax.scan(outer_step_fn, _initial_state, (dts, commands_to_apply))

    return jnp.vstack((_initial_state, states_history))

In [ ]:
def generate_param_grid(base_params, _sweep_config):
    keys = list(_sweep_config.keys())
    value_lists = []

    for key in keys:
        config = _sweep_config[key]
        start, stop, step = config['range']
        vals = jnp.arange(start, stop + step / 2, step).tolist()

        if 'index' in config:
            _idx = config['index']
            if not isinstance(_idx, tuple):
                _idx = (_idx,)

            current_arr = getattr(base_params, key)
            modified_arrs = [current_arr.at[_idx].set(v) for v in vals]
            value_lists.append(modified_arrs)
        else:
            value_lists.append(vals)

    combinations = itertools.product(*value_lists)
    param_objects = []

    for combo in combinations:
        updates = dict(zip(keys, combo))
        param_objects.append(base_params._replace(**updates))

    return param_objects

In [ ]:
_params = SystemParams(
    MASS=0.0404,
    G_WORLD=jnp.array([0.0, 0.0, -9.806599617004395]),
    I_TENSOR_DIAGONAL=jnp.array([1e-4, 1e-4, 3e-4]),
    DRAG=jnp.array([0.24, 0.4, 0.25]),
    THRUST_CONSTANT=jnp.array([0.0, 0.0, 0.06999999284744263]),
    MAX_THRUST=jnp.array([0.0, 0.0, 0.375]),
    GROUND_EFFECT_MAX=0.5,
    ROTOR_RADIUS=0.1,
    MAX_TAIL_THRUST=jnp.array([0.0, 0.0, 0.06]),
    TAIL_MOMENT_ARM=0.12,
    GYRO_SPRING_CONSTANT=jnp.array([0.01, 0.08, 0.0]),
    ANGULAR_DRAG=jnp.array([0.05, 0.025, 1e-4]),
    CORIOLIS_CONSTANT=0.06,
    ROTOR_TIME_CONSTANT=0.24,
    PITCH_TIME_CONSTANT=0.004999999888241291,
    YAW_TIME_CONSTANT=0.03
)

sweep_config = {
    'DRAG': {
        'index': 2,
        'range': (0.2, 0.4, 0.05)
    },
    # 'I_TENSOR_DIAGONAL': {
    #     'index': 1,
    #     'range': (1e-4, 1e-3, 1e-4)
    # },
    'THRUST_CONSTANT': {
        'index': 2,
        'range': (0.02, 0.07, 0.01)
    },
    'MAX_THRUST': {
        'index': 2,
        'range': (0.37, 0.39, 0.002)
    },
    'PITCH_TIME_CONSTANT': {
        'range': (0.0, 0.015, 0.005),
    },
    'MAX_TAIL_THRUST': {
        'index': 2,
        'range': (0.05, 0.12, 0.01)
    },
    'GYRO_SPRING_CONSTANT': {
        'index': 1,
        'range': (0.07, 0.09, 0.005)
    },
}

In [ ]:
params_grid = generate_param_grid(_params, sweep_config)

In [ ]:
len(params_grid)

In [ ]:
batched_params = jax.tree_util.tree_map(lambda *args: jnp.stack(args), *params_grid)

In [ ]:
@dataclass
class ParamGroupPair:
    error: float
    item: SystemParams = field(compare=False)

    def __lt__(self, other):
        return self.error > other.error

class ParamQueue:
    def __init__(self, max_size: int):
        self.max_size = max_size
        self._heap = []

    def __iter__(self):
        for queue_item in sorted(self._heap, key=lambda x: x.error):
            yield queue_item

    def __getitem__(self, item):
        return sorted(self._heap, key=lambda x: x.error)[item]

    def add(self, item: ParamGroupPair):
        if len(self._heap) < self.max_size:
            heapq.heappush(self._heap, item)
        elif item.error < self._heap[0].error:
            heapq.heapreplace(self._heap, item)

    def get_all(self):
        return sorted(self._heap, key=lambda x: x.error)

In [ ]:
_commands = jnp.array(df['command'].tolist())
swapped_commands = _commands[:, [0, 2, 1]]
swapped_commands = swapped_commands.at[:, 2].set(0.)
timestamps = jnp.array(df['timestamp'].tolist())
target = np.array(df['position'].tolist())
measured_jax = jnp.array(target)

In [ ]:
@jax.jit
def evaluate_single_param(params):
    out = run_simulation(initial_state, timestamps, params, swapped_commands, sim_fps=300)
    simulated_subset = out[:, 5:7]
    measured_subset = measured_jax[:, 1:3]
    return jnp.linalg.norm(measured_subset - simulated_subset, axis=1).mean()

batched_evaluate = jax.vmap(evaluate_single_param)

errors = batched_evaluate(batched_params)

errors_np = np.array(errors)

In [ ]:
top_n = 50
top_indices = np.argsort(errors_np)[:top_n]

best_params_queue = ParamQueue(top_n)
for idx in top_indices:
    best_params_queue.add(ParamGroupPair(errors_np[idx], params_grid[idx]))

In [ ]:
best_params_queue[0].error

In [ ]:
sim_out = run_simulation(initial_state, timestamps, best_params_queue[0].item, swapped_commands, sim_fps=500)

In [ ]:
simulated = np.array(sim_out[:, IDX_P])
measured = np.array(df['position'].tolist())

In [ ]:
sim_out.shape

In [ ]:
np.linalg.norm(simulated - measured, axis=1).mean()

In [ ]:
%matplotlib ipympl
plt.close('all')

plot_trajectories(measured_traj=measured, simulated_traj=simulated)

In [ ]:
best_params_queue[0].item

In [ ]:
sweep_config

In [ ]:
from matplotlib.animation import FuncAnimation
from scipy.spatial.transform import Rotation as sciRotation

class SimulationAnimator:
    def __init__(self):
        self.arrow = None
        self.T_IDX = 0
        self.CMD_IDX = slice(1, 4)
        self.ARROW_IDX = slice(4, 10)

    @staticmethod
    def arrow_generator(time_seq, state_seq, cmd_seq):
        i = 0
        heading = np.array([1.0, 0.0, 0.0])
        while i < len(time_seq):
            state = state_seq[i]
            cmd = cmd_seq[i]

            x, y, z = state[IDX_P]
            q_x, q_y, q_z, q_w = state[IDX_Q]

            r = sciRotation.from_quat([q_x, q_y, q_z, q_w])
            rotated_heading = r.apply(heading)

            yield (time_seq[i],
                   cmd[0], cmd[1], cmd[2],
                   x, y, z, *rotated_heading)
            i += 1

    def parse_frame_data(self, output):
        t = output[self.T_IDX]
        pos = output[self.ARROW_IDX][:3]
        cmd = output[self.CMD_IDX]
        arrow_args = output[self.ARROW_IDX]

        return (f"{t:.3f}",
                f"x={pos[0]:.3f} \ny={pos[1]:.3f} \nz={pos[2]:.3f}",
                f"thrust_cmd={cmd[0]:.3f} \npitch_cmd={cmd[1]:.3f} \nyaw_cmd={cmd[2]:.3f}",
                arrow_args)

    def plot_trajectory(self, time_seq, state_seq, cmd_seq, measured):
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        ax.set_xlim([-0.5, 0.5])
        ax.set_ylim([-1, 1])
        ax.set_zlim([-0.1, 0.5])

        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')

        positions = state_seq[:, IDX_P]
        ax.plot(positions[:, 0], positions[:, 1], positions[:, 2], 'b--', linewidth=0.5, alpha=0.5)
        ax.plot(measured[:, 0], measured[:, 1], measured[:, 2], color='red', label='Measured Trajectory')

        generator = self.arrow_generator(time_seq, state_seq, cmd_seq)
        first_frame = self.parse_frame_data(next(generator))

        self.arrow = ax.quiver(*first_frame[3], color='b', length=0.75, pivot='middle', zorder=0)
        timestamp = ax.text2D(-0.25, 0.0, f"t = {first_frame[0]}", transform=ax.transAxes)
        pos_text = ax.text2D(-0.25, 1.0, f"{first_frame[1]}", transform=ax.transAxes)
        cmd_text = ax.text2D(0.9, 1.0, f"{first_frame[2]}", transform=ax.transAxes)

        def update(frame):
            parsed = self.parse_frame_data(frame)
            timestamp.set_text(f't = {parsed[0]}')
            pos_text.set_text(f'{parsed[1]}')
            cmd_text.set_text(f'{parsed[2]}')

            self.arrow.remove()
            self.arrow = ax.quiver(*parsed[3], color='b', length=0.2, pivot='middle', zorder=0)

            return (self.arrow,)

        ani = FuncAnimation(fig, update, frames=generator, interval=40, blit=False, cache_frame_data=False)

        return ani

In [ ]:
ani = SimulationAnimator()

In [ ]:
%matplotlib ipympl
plt.close('all')
ani.plot_trajectory(np.array(df['timestamp'].tolist()),
                    np.asarray(sim_out),
                    np.asarray(swapped_commands),
                    measured)